# Loan Limit Increase — подробный разбор (RU)

Этот ноутбук дублирует ключевую логику основного EDA, но с более подробными комментариями.

Цель: пошагово показать, как из сырых данных построить решение для оптимизации увеличений лимита:
- EDA и проверка качества данных
- Сегментация риска (Prime / Near-Prime / Sub-Prime)
- Марковская модель миграции риска
- Модель вероятности принятия оффера
- Простая оптимизация портфеля
- Монте-Карло оценка распределения NPV

## 0. Импорт библиотек и параметры

Ниже задаются глобальные константы и параметры модели.
- `PROFIT_PER_INCREASE=40` — доход на 1 увеличение лимита
- `ANNUAL_DISCOUNT_RATE=0.19` — ставка для NPV
- `ELIGIBILITY_DAYS=60` — правило eligibility из задания
- `MAX_INCREASES_PER_YEAR=6` — лимит из задания

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import linprog
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings("ignore")

# Пути: предполагаем запуск из папки notebooks/
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "loan_limit_increases.csv"

# Бизнес-константы
PROFIT_PER_INCREASE = 40
ANNUAL_DISCOUNT_RATE = 0.19
MONTHLY_DISCOUNT_RATE = (1 + ANNUAL_DISCOUNT_RATE) ** (1 / 12) - 1
ELIGIBILITY_DAYS = 60
MAX_INCREASES_PER_YEAR = 6
CAPITAL_LIMIT_RATIO = 0.30

SEED = 42
np.random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")

print("Environment ready. Project root:", PROJECT_ROOT)

## 1. Загрузка данных и первичная проверка

На этом шаге:
1. Читаем CSV.
2. Нормализуем имена колонок, чтобы дальше код читался проще.
3. Проверяем типы, пропуски и базовые статистики.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = [
    "customer_id",
    "initial_loan",
    "days_since_last_loan",
    "on_time_pct",
    "n_increases_2023",
    "total_profit",
]

print("Размер данных:", df.shape)
print("\nТипы колонок:")
print(df.dtypes)
print("\nПропуски:")
print(df.isna().sum())
print("\nПервые строки:")
df.head()

In [ ]:
# Быстрый sanity-check на логику прибыли
df["profit_rule_check"] = np.maximum(0, df["n_increases_2023"] - 2) * PROFIT_PER_INCREASE
mismatch = (df["profit_rule_check"] != df["total_profit"]).sum()
print("Несовпадений с правилом max(0, n-2)*40:", mismatch)

df[["n_increases_2023", "total_profit", "profit_rule_check"]].head(10)

## 2. EDA: распределения и ключевые зависимости

Здесь смотрим:
- кто проходит eligibility по 60 дням,
- как связаны on-time платежи и факт увеличения лимита,
- базовые распределения признаков.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["days_since_last_loan"], bins=40, color="steelblue", alpha=0.85)
axes[0].axvline(ELIGIBILITY_DAYS, color="red", linestyle="--", label="60 дней")
axes[0].set_title("Days since last loan")
axes[0].legend()

axes[1].hist(df["on_time_pct"], bins=40, color="seagreen", alpha=0.85)
axes[1].set_title("On-time payment %")

inc_counts = df["n_increases_2023"].value_counts().sort_index()
axes[2].bar(inc_counts.index.astype(str), inc_counts.values, color="darkorange")
axes[2].set_title("Число увеличений в 2023")

plt.tight_layout()
plt.show()

pct_eligible = (df["days_since_last_loan"] >= ELIGIBILITY_DAYS).mean() * 100
pct_received = (df["n_increases_2023"] > 0).mean() * 100
print(f"Eligible (>=60 days): {pct_eligible:.1f}%")
print(f"Received >=1 increase: {pct_received:.1f}%")

## 3. Feature Engineering + риск-сегменты

Правила сегментации:
- Prime: `on_time_pct >= 95`
- Near-Prime: `85 <= on_time_pct < 95`
- Sub-Prime: `< 85`

Дополнительно создаём:
- `eligible`
- `max_possible_increases`
- `utilisation_rate`
- `accepted` (таргет для логистической регрессии)

In [ ]:
def risk_state(on_time_pct: float) -> int:
    if on_time_pct >= 95:
        return 0  # Prime
    if on_time_pct >= 85:
        return 1  # Near-Prime
    return 2      # Sub-Prime

df["risk_state"] = df["on_time_pct"].apply(risk_state)
df["risk_label"] = df["risk_state"].map({0: "Prime", 1: "Near-Prime", 2: "Sub-Prime"})
df["eligible"] = (df["days_since_last_loan"] >= ELIGIBILITY_DAYS).astype(int)

# Максимум теоретических увеличений по календарному ограничению
df["max_possible_increases"] = (
    (df["days_since_last_loan"].clip(upper=365) // ELIGIBILITY_DAYS)
    .clip(upper=MAX_INCREASES_PER_YEAR)
    .astype(int)
)

# Нормировка размера займа как proxy уровня экспозиции
df["utilisation_rate"] = (df["initial_loan"] - df["initial_loan"].min()) / (df["initial_loan"].max() - df["initial_loan"].min())

# Таргет принятия оффера
df["accepted"] = (df["n_increases_2023"] > 0).astype(int)

df[["risk_label", "eligible", "max_possible_increases", "utilisation_rate", "accepted"]].head()

## 4. Марковская модель миграции риска

Поскольку в датасете один срез (а не история переходов по времени), матрицу переходов задаём как калиброванное допущение.

Интерпретация матрицы `P`:
- строка = текущее состояние
- столбец = состояние в следующем 60-дневном периоде

In [ ]:
P = np.array([
    [0.88, 0.10, 0.02],  # Prime -> ...
    [0.08, 0.82, 0.10],  # Near-Prime -> ...
    [0.03, 0.12, 0.85],  # Sub-Prime -> ...
])

# Годовые вероятности дефолта по сегментам риска
P_DEFAULT_ANNUAL = np.array([0.02, 0.06, 0.15])

assert np.allclose(P.sum(axis=1), 1.0), "Строки матрицы переходов должны суммироваться к 1"

display(pd.DataFrame(P, index=["Prime", "Near-Prime", "Sub-Prime"], columns=["to Prime", "to Near-Prime", "to Sub-Prime"]))
print("PD annual by state:", P_DEFAULT_ANNUAL)

## 5. Модель спроса: вероятность принятия оффера

Используем `LogisticRegression`, где таргет: `accepted` (получил хотя бы 1 увеличение).

Важно: в реальном проде таргет лучше формировать на уровне `offer -> response`, если доступна история офферов.

In [ ]:
features = [
    "days_since_last_loan",
    "on_time_pct",
    "initial_loan",
    "risk_state",
    "utilisation_rate",
]
X = df[features].values
y = df["accepted"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_train_s, y_train)

pred = clf.predict(X_test_s)
proba = clf.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))

# Оценка вероятности принятия для всего портфеля
df["p_accept"] = clf.predict_proba(scaler.transform(df[features].values))[:, 1]
df[["customer_id", "p_accept"]].head()

## 6. Простая портфельная оптимизация (LP)

Идея: выбираем число увеличений `x_i` для каждого клиента так, чтобы максимизировать ожидаемую прибыль при ограничении на капитал.

Здесь решаем LP-релаксацию (`x_i` непрерывные), потом округление можно делать отдельно (greedy / branch-and-bound).

In [ ]:
state_vec = df["risk_state"].values
p_def = P_DEFAULT_ANNUAL[state_vec]
p_acc = df["p_accept"].values

# Средний discount-фактор для 6 потенциальных 60-дневных периодов
pv_factors = np.array([1 / (1 + MONTHLY_DISCOUNT_RATE) ** (2 * k) for k in range(1, 7)])
avg_pv = pv_factors.mean()

expected_profit_per_inc = p_acc * (1 - p_def) * PROFIT_PER_INCREASE * avg_pv

x_upper = df["max_possible_increases"].clip(upper=MAX_INCREASES_PER_YEAR).astype(float).values
x_upper[df["eligible"].values == 0] = 0.0

capital_cost = df["initial_loan"].values * p_acc
capital_budget = CAPITAL_LIMIT_RATIO * (capital_cost.sum())

result = linprog(
    c=-expected_profit_per_inc,
    A_ub=capital_cost.reshape(1, -1),
    b_ub=np.array([capital_budget]),
    bounds=[(0, x_upper[i]) for i in range(len(df))],
    method="highs",
)

if result.status == 0:
    x_lp = result.x
    print("LP solved. Expected portfolio profit:", round(-result.fun, 2))
    print("Capital usage (%):", round((capital_cost * x_lp).sum() / capital_budget * 100, 2))
else:
    x_lp = np.zeros(len(df))
    print("LP failed:", result.message)

## 7. Монте-Карло для распределения NPV

На каждой симуляции:
1. Для каждого клиента бросаем Bernoulli на принятие оффера.
2. Для принявших бросаем Bernoulli на дефолт (упрощённо по period PD).
3. Считаем `profit - loss` и суммируем по портфелю.

Так получаем не одну точку, а распределение NPV: среднее, риск, хвосты.

In [ ]:
def monte_carlo_npv(x_alloc, p_accept, p_default, loan_amounts, n_sim=3000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(x_alloc)

    # Перевод годовой PD в период 60 дней (грубое приближение)
    pd_period = p_default / 6

    accept = rng.random((n_sim, n)) < p_accept[np.newaxis, :]
    default = rng.random((n_sim, n)) < pd_period[np.newaxis, :]

    x = x_alloc[np.newaxis, :]
    profit = PROFIT_PER_INCREASE * avg_pv * x * (accept & ~default)
    loss = 0.5 * loan_amounts[np.newaxis, :] * (accept & default)

    return (profit - loss).sum(axis=1)

npv_dist = monte_carlo_npv(
    x_alloc=x_lp,
    p_accept=p_acc,
    p_default=p_def,
    loan_amounts=df["initial_loan"].values,
)

print("E[NPV]:", round(npv_dist.mean(), 2))
print("Std NPV:", round(npv_dist.std(), 2))
print("VaR(5%):", round(np.percentile(npv_dist, 5), 2))

plt.figure(figsize=(8, 4))
plt.hist(npv_dist, bins=70, color="slateblue", alpha=0.8)
plt.axvline(npv_dist.mean(), color="black", linestyle="--", label="Mean")
plt.title("Распределение NPV (Monte Carlo)")
plt.xlabel("Portfolio NPV")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Что важно улучшить перед production

1. **История офферов**: перейти от прокси-таргета (`n_increases_2023 > 0`) к факту `offer -> accept/reject`.
2. **Калибровка PD/LGD**: обучать на фактических дефолтах и recovery.
3. **Backtesting**: прогон на out-of-time периодах (например, train до Q3, test на Q4).
4. **Fairness/Compliance**: аудит признаков и регулярный drift-monitoring.
5. **Decisioning pipeline**: nightly scoring + журнал решений для аудита регулятором.